In [1]:
import torch
import math
import numpy as np
import copy

from tokenizers import Tokenizer
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path

In [2]:
torch.manual_seed(0)

In [3]:
TOKENIZER = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json"

SFT_TRAIN_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_tokens.bin"
SFT_TRAIN_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_mask.bin"
SFT_TRAIN_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_train_offsets.npy"

SFT_VALIDATION_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_tokens.bin"
SFT_VALIDATION_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_mask.bin"
SFT_VALIDATION_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_valid_offsets.npy"

SFT_TEST_TOKENS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_tokens.bin"
SFT_TEST_MASK = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_mask.bin"
SFT_TEST_OFFSETS = "/kaggle/input/datasets/absolute1000/tokenized-llm-training/sft_test_offsets.npy"

BEST_SFT_MODEL = '/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt'

tok = Tokenizer.from_file(TOKENIZER)
USER_ID = tok.token_to_id("<user>")
BOT_ID = tok.token_to_id("<bot>")
PAD_ID = tok.token_to_id("<pad>")
MASK_ID = tok.token_to_id("<mask>")
EOS_ID = tok.token_to_id("<eos>")
print(USER_ID, BOT_ID, PAD_ID, MASK_ID, EOS_ID)

2 3 0 1 4


PREVIOUS CODE: DATA
===

In [4]:
class SFTDataset(torch.utils.data.Dataset):

    def __init__(self, tokens_path, mask_path, offsets_path):
        self.tokens_path = tokens_path
        self.mask_path = mask_path
        self.offsets_path = offsets_path
        self.tokens = np.memmap(self.tokens_path, dtype=np.uint16, mode="r")
        self.mask = np.memmap(self.mask_path, dtype=np.uint8, mode="r")
        self.offsets = np.load(self.offsets_path)
        self.length = len(self.offsets) - 1

    def __getitem__(self, index):
        li = self.offsets[index]
        ri = self.offsets[index + 1]

        dialogue = self.tokens[li:ri]
        mask = self.mask[li:ri]

        temp_helper = np.diff(mask)
        bot_phrases_starts_idx = np.where(temp_helper==1)[0]
        bot_phrases_starts_idx += np.ones(len(bot_phrases_starts_idx), dtype=np.int64)
        bot_phrases_ends_idx = np.where(temp_helper==255)[0]
        bot_phrases_ends_idx += np.ones(len(bot_phrases_ends_idx), dtype=np.int64)
        
        random_start = bot_phrases_starts_idx[torch.randint(len(bot_phrases_starts_idx), (1,)).item()]
        end_of_chosen_bot_phrase_idx = np.where(bot_phrases_ends_idx>random_start)[0]
        
        context = dialogue[:random_start]
        context_mask = mask[:random_start]
        if end_of_chosen_bot_phrase_idx.size == 0:
            bot_tokens = dialogue[random_start:]
            bot_mask = mask[random_start:]
        else:
            idx = end_of_chosen_bot_phrase_idx[0]
            bot_tokens = dialogue[random_start:bot_phrases_ends_idx[idx]]
            bot_mask = mask[random_start:bot_phrases_ends_idx[idx]]

        context = torch.from_numpy(context.astype(np.int64))
        bot_tokens = torch.from_numpy(bot_tokens.astype(np.int64))
        return context, bot_tokens
    def __len__(self):
        return self.length

def collate_fn(list_of_samples):
    #tok.encode("<pad>").ids = [0]
    
    B = len(list_of_samples)
    _batch_samples = torch.zeros((B, 1000), dtype=torch.int64)
    _batch_targets = torch.zeros((B, 500), dtype=torch.int64)
    for idx, data in enumerate(list_of_samples):
        context, bot_tokens = data
        _batch_samples[idx, -len(context):] = context
        _batch_targets[idx, :len(bot_tokens)] = bot_tokens
    return _batch_samples, _batch_targets

PREVIOUS CODE: MODEL
===

In [5]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [6]:
MODEL_SETTINGS_CONFIG = {
    "embedding_size": 384,
    "encoder_num": 4,
    "heads_encoder": 6,
    "thinking_internal_state_size": 128,
    "decoder_num": 2,
    "heads_decoder": 6,
}

In [7]:
from collections import Counter

text = np.memmap(SFT_TRAIN_TOKENS, dtype=np.uint16, mode="r")
is_bot = np.memmap(SFT_TRAIN_MASK, dtype=np.uint8, mode="r")

d = np.diff(is_bot.astype(np.int8), prepend=0, append=0)
starts, ends = np.where(d == 1)[0], np.where(d == -1)[0]

answers = Counter(tuple(text[s:e - 1].tolist()) for s, e in zip(starts, ends))   # e − 1: без <eos>
top = answers.most_common(50)
FORBIDDEN_PHRASES = {a for a, _ in top}

for i, (a, c) in enumerate(top):
    print(f"{i}: {c:5d}  {tok.decode(list(a))!r}")

0:   172  'Thank you.'
1:   115  'What do you do for a living?'
2:   107  'Yes.'
3:    77  'What happened?'
4:    67  'OK.'
5:    67  'Here you are.'
6:    64  'Thanks.'
7:    64  'Why?'
8:    58  "You're welcome."
9:    57  'Thank you very much.'
10:    55  'Do you have any hobbies?'
11:    51  'Sure.'
12:    51  'Why is that?'
13:    47  'Really?'
14:    46  'No problem.'
15:    38  'What do you mean?'
16:    38  'What do you do for work?'
17:    34  'What city are you from?'
18:    33  'All right.'
19:    33  'Why not?'
20:    32  'What?'
21:    32  'Do you have any pets?'
22:    29  'I see.'
23:    29  'What did you do?'
24:    29  'What is it?'
25:    26  'Thanks a lot.'
26:    26  "That's right."
27:    26  'Of course.'
28:    26  'Hi how are you today.'
29:    25  'No.'
30:    25  'Not at all.'
31:    24  'You are welcome.'
32:    23  "I'm sorry to hear that."
33:    23  'Yes, please.'
34:    23  'thats cool'
35:    22  'Certainly.'
36:    22  'No, thanks.'
37:    22  "Don't men

In [8]:
tokens_split = ["train", "valid", "test"]
DATA_DIR = "/kaggle/input/datasets/absolute1000/tokenized-llm-training"

ctx_all, ans_all = [], []
for split in tokens_split:
    m = np.fromfile(f"{DATA_DIR}/sft_{split}_mask.bin", dtype=np.uint8)
    off = np.load(f"{DATA_DIR}/sft_{split}_offsets.npy")
    d = np.diff(m.astype(np.int8), prepend=0, append=0)
    s, e = np.where(d == 1)[0], np.where(d == -1)[0]
    dialog_start = off[np.searchsorted(off, s, side="right") - 1]
    ctx_all.append(np.minimum(s - dialog_start, 256))
    ans_all.append(e - s - 1)

ctx_len = np.concatenate(ctx_all)
ans_len = np.concatenate(ans_all)

edges = np.arange(0, 270, 10)
idx = np.digitize(ctx_len, edges) - 1
LEN_STATS = {}
print(f"{'контекст':>10} {'ответов':>8} {'q5':>6} {'q10':>6} {'q25':>6} {'среднее':>8} {'медиана':>8} "
      f"{'q75':>6} {'q95':>6} {'мин':>5}")
for b in range(len(edges) - 1):
    sel = idx == b
    if sel.any():
        a = ans_len[sel]
        q5, q10, q25, q75, q95 = np.percentile(a, [5, 10, 25, 75, 95])
        LEN_STATS[int(edges[b])] = [float(q5), float(q10), float(q25), float(a.mean()), int(a.min()),
                                    float(q75), float(q95)]
        print(f"{edges[b]:>4}-{edges[b + 1]:<5} {sel.sum():>8} {q5:>6.1f} {q10:>6.1f} {q25:>6.1f} "
              f"{a.mean():>8.1f} {np.median(a):>8.0f} {q75:>6.1f} {q95:>6.1f} {a.min():>5}")
print(f"\nвсего ответов: {len(ans_len)}, средний ответ: {ans_len.mean():.1f}")

  контекст  ответов     q5    q10    q25  среднее  медиана    q75    q95   мин
   0-10        6131    4.0    6.0    8.0     12.1       11   15.0   22.0     1
  10-20       23763    5.0    6.0    9.0     13.9       12   17.0   27.0     1
  20-30       13046    5.0    6.0    9.0     15.5       13   18.0   32.0     1
  30-40       12528    5.0    6.0    9.0     14.1       13   17.0   28.0     1
  40-50       13029    5.0    6.0    9.0     14.7       14   18.0   28.0     1
  50-60       10746    5.0    7.0   10.0     15.4       14   19.0   30.0     1
  60-70        8945    6.0    7.0   10.0     15.7       14   19.0   31.0     1
  70-80        7886    6.0    7.0   10.0     16.1       15   19.0   32.0     1
  80-90        6873    6.0    7.0   10.0     16.0       15   19.0   31.0     2
  90-100       5715    6.0    7.0   10.0     16.0       15   19.0   33.0     1
 100-110       4988    6.0    7.0   10.0     15.4       14   18.0   31.0     1
 110-120       4792    6.0    7.0   10.0     15.7   

In [9]:
@torch.no_grad()
def rollout(model, batch_context, attempts_num, answer_len=64, temperature=1.0, top_k=None, block_size=8,
            min_len=3, use_blocks=True):
    device = batch_context.device
    N = batch_context.shape[0] * attempts_num
    KV_cache, mask = model.encode(batch_context) #[B, Cs, Emb], [B, 1, Cs]
    KV_cache = KV_cache.repeat_interleave(attempts_num, dim=0) # [B * attempts_num, Cs, Emb]
    mask = mask.repeat_interleave(attempts_num, dim=0) #[B * attempts_num, 1, Cs]
    answer_ids = torch.full((batch_context.shape[0] * attempts_num, answer_len), MASK_ID, dtype=torch.long, device=device) # [B * attempts_num, answer_size]
    
    pos = torch.arange(answer_ids.shape[1], device=device) # [answer_size]
    rows = torch.arange(N, device=device)
    
    while True:
        is_mask = answer_ids == MASK_ID #  [B * attempts_num, answer_size]
        if not is_mask.any():
            break

        logits = model.decode(answer_ids, KV_cache, mask).float() # [B * attempts_num, answer_size, dict_size]
        has_eos = (answer_ids == EOS_ID).any(-1) # [B * attempts_num]
        logits[..., [PAD_ID, MASK_ID]] = -float("inf")
        logits[:, :min_len, EOS_ID] = -float("inf")
        logits[has_eos, :, EOS_ID] = -float("inf")

        editable = is_mask.clone() # [B * attempts_num, answer_size]
        if use_blocks:
            first = is_mask.int().argmax(-1) # [B * attempts_num]
            editable &= pos[None, :] < (first[:, None] // block_size + 1) * block_size

        logits = logits / temperature
        if top_k is not None:
            treshold = logits.topk(top_k, dim=-1).values[:, :, -1:] # [B * attempts_num, answer_size, 1]
            probs = torch.softmax(logits.masked_fill(logits < treshold, -float("inf")), -1) # [B * attempts_num, answer_size, dict_size]
        else:
            probs = torch.softmax(logits, -1)
        pred = torch.multinomial(probs.view(-1, probs.shape[-1]), 1).view(N, answer_len, 1) # [B * attempts_num, answer_size, 1]
        choice_confidence = probs.gather(-1, pred).squeeze(-1) # [B * attempts_num, answer_size, 1] -> [B * attempts_num, answer_size]
        choice_confidence = choice_confidence.masked_fill(~editable, -1.0)
        i = choice_confidence.argmax(dim=-1) # [B * attempts_num]
        
        active = is_mask.any(-1)
        update = rows[active]
        answer_ids[update, i[update]] = pred[update, i[update], 0]

        is_eos = answer_ids == EOS_ID
        eos_pos = torch.where(is_eos.any(-1), is_eos.int().argmax(-1), torch.full_like(rows, answer_len))
        answer_ids[pos[None, :] > eos_pos[:, None]] = PAD_ID
    is_eos = answer_ids == EOS_ID
    L = torch.where(is_eos.any(-1), is_eos.int().argmax(-1), torch.full_like(rows, answer_len))
    return answer_ids, L

In [10]:
@torch.no_grad()
def reward(ctx_ids, ans_ids, L, DEVICE, answer_window=64):
    # ctx_ids [N, Cs], ans_ids [N, La]. L [N]
    ctx_ids, ans_ids, L = ctx_ids.cpu(), ans_ids.cpu(), L.cpu()
    R = torch.zeros(L.shape[0], dtype=torch.float)
    stats = {
        'length': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'no_end': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'repeat': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'generic': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'copy': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'start_punct': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device),
        'end_punct': torch.zeros(L.shape[0], dtype=torch.float, device=ctx_ids.device)
        }
    for sample in range(L.shape[0]):
        ans = ans_ids[sample, :L[sample]].tolist()
        ans_txt = tok.decode(ans)
        
        ctx_raw = ctx_ids[sample]
        if len((ctx_raw == USER_ID).nonzero(as_tuple=True)[0]) == 0:
            left = 0
        else:
            left = (ctx_raw == USER_ID).nonzero(as_tuple=True)[0][-1]
        right = (ctx_raw == BOT_ID).nonzero(as_tuple=True)[0][-1]
        question = ctx_raw[left+1:right].tolist()
        
        question_txt = tok.decode(question)

        #1
        k = min(int((ctx_raw != PAD_ID).sum()) // 10 * 10, 250)
        l = LEN_STATS.get(k, [4,0,0,0,0,0,60])
        if len(ans) < l[0] or len(ans) > l[6]:
            R[sample] -= 1
            stats['length'][sample] -= 1
        #2
        if len(ans) == answer_window:
            R[sample] -= 1
            stats['no_end'][sample] -= 1
        #3
        if tuple(ans) in FORBIDDEN_PHRASES:
            R[sample] -= 1
            stats['generic'][sample] -= 1

        #4
        trigram_ans = [tuple(ans[j:j + 3]) for j in range(len(ans) - 2)]
        trigram_question = [tuple(question[j:j + 3]) for j in range(len(question) - 2)]
        
        s1, s2 = set(trigram_ans), set(trigram_question)
        if len(ans) > 4 and len(s1.intersection(s2))/max(len(trigram_ans), 1) > 0.5:
            R[sample] -= 1
            stats['copy'][sample] -= 1

        #5
        if (a := ans_txt.lstrip()) != '' and a[0] in ['.', ',', '!', '?', ';', ':']:
            R[sample] -= 0.5
            stats['start_punct'][sample] -= 0.5

        #6
        if (a := ans_txt.rstrip()) != '' and a[-1] in ['.', '!', '?']:
            R[sample] += 0.5
            stats['end_punct'][sample] += 0.5

        #7
        repeats = (len(trigram_ans) - len(set(trigram_ans))) / max(len(trigram_ans), 1)
        if repeats > 0.2:
            R[sample] += -3 * repeats
            stats['repeat'][sample] += -3 * repeats
    return R.to(DEVICE), stats

In [11]:
@torch.no_grad()
def reward_norming(R, attemps_num):
    samples = R.shape[0] // attemps_num
    R = R.reshape(samples, attemps_num).float()
    mean = R.mean(dim=1, keepdim=True)
    std = R.std(dim=1, keepdim=True)
    valid_mask = (std > 1e-4).repeat_interleave(attemps_num)
    R = (R - mean) / (std + 1e-6)
    return R.reshape(samples * attemps_num), valid_mask    

In [12]:
@torch.no_grad()
def make_noise(N, ans_size, K, device):
    pairs = []
    for _ in range(K):
        t = ((torch.arange(N, device=device) + torch.rand(1, device=device))/N).clamp(min=0.1)
        t = t[torch.randperm(N, device=device)] # [N]
        noise = torch.rand(N, ans_size, device=device) < t[:, None]
        pairs.append((t, noise))
    return pairs

In [13]:
def seq_logprob(model, memory, mask, ans, pairs):
    # memory[N, Cs, E], mask[N, 1, Cs], ans[N, ans_size], noise_list[N, ans_size], t[N]
    model.eval()
    policy_log = torch.zeros(mask.shape[0], device=mask.device)
    for t, noise in pairs:
        masked = torch.where(noise, MASK_ID, ans) #[N, ans_size]
        logits = model.decode(masked, memory, mask).float().transpose(1, 2) # [N, answer_size, dict_size] -> [N, dict_size, answer_size]
        cross_entropy = torch.nn.functional.cross_entropy(logits, ans, reduction='none') #[N, answer_size]
        loss = -torch.where(noise, cross_entropy, 0).sum(dim=-1)/t #[N]
        policy_log += loss
    return policy_log / len(pairs) / ans.shape[1]

In [14]:
def grpo_loss(logp_rl, logp_sft, A, valid, eps=0.2, beta=0.04):
    #logp_new [N], logp_old [N], logp_ref [N], A [N], valid [N], policy_log, stats
    loss_rl = -(A[valid] * logp_rl[valid]).sum() / valid.sum()
    diff = logp_sft - logp_rl
    kl = (torch.exp(diff) - diff - 1)[valid].mean()
    loss = loss_rl + beta * kl
    return loss, kl.item()

In [15]:
def make_record(reason, step, model_rl, optimizer, scheduler_warmup, scaler, best_mean_reward, context_size, answer_size, config,
               rl_loss_history, val_loss_history):
    path = f'/kaggle/working/RL_{reason}.pt'
    Path(path).unlink(missing_ok=True)
    torch.save({
        'step': step,
        'model': model_rl.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler_warmup.state_dict(),
        'scaler': scaler.state_dict(),
        'best_mean_reward': best_mean_reward,
        'context_size': context_size,
        'answer_size': answer_size,
        'rl_loss_history': rl_loss_history,
        'val_loss_history': val_loss_history,
        'MODEL_SETTINGS_CONFIG': MODEL_SETTINGS_CONFIG
    }, path)

In [16]:
def REINFORCEMENT_LEARNING_CYCLE(
    init_path=None,
    resume_path=None,
    steps_num=10000,
    start_step=0,
    attempts_num=8,
    batch_size=16,
    context_size=256,
    answer_size=64,
    weight_decay = 0.01,
    warmup_steps=20,
    lr = 2e-6,
#
    rollout_temperature=1,
    rollout_top_k=None,
    rollout_block_size=8,
    rollout_use_blocks=True,
    rollout_min_len=3,
#
    noising_num=2,
    eps=0.2,
    beta=0.04,
    best_mean_reward=-float('inf'),
#
    MODEL_SETTINGS_CONFIG=MODEL_SETTINGS_CONFIG
):
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    def infinite(loader):
        while True:
            for batch in loader:
                yield batch
    data = infinite(
        torch.utils.data.DataLoader(
            SFTDataset(SFT_TRAIN_TOKENS, SFT_TRAIN_MASK, SFT_TRAIN_OFFSETS), batch_size, shuffle=True, num_workers=4,
            collate_fn=collate_fn, pin_memory=True, drop_last=True, persistent_workers=True
        )
    )
    VAL_TEXTS = [
        ("What is machine learning?", "NOTHING"),
        ("Why is my loss not going down?", "NOTHING"),
        ("What is attention in neural nets?", "NOTHING"),
        ("How does temperature affect sampling?", "NOTHING"),
        ("What is a mask token in diffusion?", "NOTHING"),
        ("Explain overfitting simply.", "NOTHING"),
        ("What is gradient descent?", "NOTHING"),
    
        ("What is an API?", "NOTHING"),
        ("What is the difference between TCP and UDP?", "NOTHING"),
        ("What is a hash table?", "NOTHING"),
    
        ("What is a derivative?", "NOTHING"),
        ("What is Bayes' theorem?", "NOTHING"),
    
        ("How is the weather today?", "NOTHING"),
        ("What are you eating?", "NOTHING"),
        ("I am too tired to cook tonight.", "NOTHING"),
        ("How do I ask someone out?", "NOTHING"),
    ]
    PROMPTS_FOR_VAL = [
        (
            torch.tensor([USER_ID] + tok.encode(prompt[0]).ids + [BOT_ID], dtype=torch.long), 
            torch.tensor((tok.encode(prompt[1])).ids, dtype=torch.long)
        )
    for prompt in VAL_TEXTS
    ]
    val_promts = collate_fn(PROMPTS_FOR_VAL)
    ckpt = torch.load(resume_path or init_path, map_location=DEVICE, weights_only=False) if (resume_path or init_path) else None
    config = ckpt['MODEL_SETTINGS_CONFIG'] if ckpt else MODEL_SETTINGS_CONFIG
    model_rl = NN(**config).to(DEVICE)
    model_rl.load_state_dict(ckpt['model'])

    decay    = [p for p in model_rl.parameters() if p.dim() >= 2]
    no_decay = [p for p in model_rl.parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": weight_decay}, {"params": no_decay, "weight_decay": 0.0}], 
                                  lr=lr)
    scaler = torch.amp.GradScaler(DEVICE)
    scheduler_warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=warmup_steps)

    model_sft = copy.deepcopy(model_rl); model_sft.requires_grad_(False)
    model_rl.eval(), model_sft.eval()
    
    rl_loss_history, val_reward_history = [], []
    RL_CYCLE = tqdm(range(start_step, steps_num), desc=f"RL CYCLE|")
    skip_counting=0
    for step in RL_CYCLE:
        batch = next(data)
        batch_context_ids = batch[0][:, -context_size:].to(DEVICE)
        with torch.amp.autocast(DEVICE), torch.no_grad():
            KV_cache_sft, mask_sft = [value.repeat_interleave(attempts_num, 0) for value in model_sft.encode(batch_context_ids)]
            
            output_rl, output_rl_len = rollout(model_rl, batch_context=batch_context_ids, attempts_num=attempts_num, answer_len=answer_size,
                                temperature=rollout_temperature, top_k=rollout_top_k, block_size=rollout_block_size, 
                                min_len=rollout_min_len, use_blocks=rollout_use_blocks)
            
            R_rl, stats_rl = reward(batch_context_ids.repeat_interleave(attempts_num, dim=0), output_rl, output_rl_len, DEVICE, 
                              answer_window=answer_size)
            A_rl, valid_mask_rl = reward_norming(R_rl, attempts_num)
            if not valid_mask_rl.any():
                skip_counting += 1
                continue

            pairs = make_noise(batch_size * attempts_num, answer_size, noising_num, DEVICE)

            sft_logpolicy = seq_logprob(model_sft, KV_cache_sft, mask_sft, output_rl, pairs)
        
        with torch.amp.autocast(DEVICE):
            KV_cache_rl, mask_rl = model_rl.encode(batch_context_ids)
            KV_cache_rl = KV_cache_rl.repeat_interleave(attempts_num, 0)
            mask_rl = mask_rl.repeat_interleave(attempts_num, 0)
            rl_logpolicy = seq_logprob(model_rl, KV_cache_rl, mask_rl, output_rl, pairs)

            loss, kl = grpo_loss(rl_logpolicy, sft_logpolicy, A_rl, valid_mask_rl, eps=eps, beta=beta)
            RL_CYCLE.set_postfix({"R": f"{R_rl.mean().item():+.2f}", "kl": f"{kl:.3f}", "loss": f"{loss.item():.3f}"})
            if step % 10 == 0:
                print(f"step {step}: R {R_rl.mean().item():+.2f}  kl {kl:.3f}  " +
                      "  ".join(f"{k} {v.mean().item():+.2f}" for k, v in stats_rl.items()))
            rl_loss_history.append(loss.item())

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model_rl.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler_warmup.step()
        if step % 50 == 0:
            make_record("RECORD", step, model_rl, optimizer, scheduler_warmup, scaler, best_mean_reward, context_size, 
                        answer_size, config, rl_loss_history, val_reward_history)

        if step % 100 == 0 or step == steps_num:
            with torch.amp.autocast(DEVICE), torch.no_grad():
                val_context_ids = val_promts[0][:, -context_size:].to(DEVICE)

                cpu_state, cuda_state = torch.random.get_rng_state(), torch.cuda.get_rng_state()
                torch.manual_seed(0)
                
                output_rl, output_rl_len = rollout(model_rl, batch_context=val_context_ids, attempts_num=attempts_num, answer_len=answer_size,
                                    temperature=rollout_temperature, top_k=rollout_top_k, block_size=rollout_block_size, 
                                    min_len=rollout_min_len, use_blocks=rollout_use_blocks)
                R_rl, stats_rl = reward(val_context_ids.repeat_interleave(attempts_num, dim=0), output_rl, output_rl_len, DEVICE, 
                                  answer_window=answer_size)                
                mean_reward = R_rl.mean().item()

                torch.random.set_rng_state(cpu_state)
                torch.cuda.set_rng_state(cuda_state)
                
                val_reward_history.append(mean_reward)
            print(f'VAL| reward {mean_reward}, SK {skip_counting}')
            skip_counting = 0
            for i in range(0, output_rl.shape[0], attempts_num):
                print(f'ANSWER TO CONTEXT{i // attempts_num}: {tok.decode(output_rl[i, :output_rl_len[i]].tolist())}')
            if mean_reward > best_mean_reward:
                best_mean_reward = mean_reward
                make_record("BEST", step, model_rl, optimizer, scheduler_warmup, scaler, best_mean_reward, context_size, answer_size, config,
                           rl_loss_history, val_reward_history)
    return model_rl

In [17]:
PROCESS_CONFIG = {
    "init_path": BEST_SFT_MODEL,
    "resume_path": None,
    "steps_num": 500,
    "start_step": 0,
    "attempts_num": 8,
    "batch_size": 16,
    "context_size": 256,
    "answer_size": 64,
    "weight_decay": 0.01,
    "warmup_steps": 20,
    "lr": 5e-6,
    "rollout_temperature": 1,
    "rollout_top_k": None,
    "rollout_block_size": 8,
    "rollout_use_blocks": True,
    "rollout_min_len":3,
    "noising_num":2,
    "eps":0.2,
    "beta":0.1,
    "best_mean_reward":-float('inf'),
    "MODEL_SETTINGS_CONFIG":MODEL_SETTINGS_CONFIG
}

In [18]:
model = REINFORCEMENT_LEARNING_CYCLE(**PROCESS_CONFIG)

RL CYCLE|:   0%|          | 0/500 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:2954: UserWarning: Mismatch dtype between input and weight: input dtype = c10::Half, weight dtype = float, Cannot dispatch to fused implementation. (Triggered internally at /pytorch/aten/src/ATen/native/layer_norm.cpp:344.)
  return torch.rms_norm(input, normalized_shape, weight, eps)
RL CYCLE|:   0%|          | 0/500 [00:04<?, ?it/s, R=-0.14, kl=0.000, loss=-0.544]

step 0: R -0.14  kl 0.000  length -0.30  no_end -0.16  repeat -0.02  generic -0.01  copy +0.00  start_punct -0.05  end_punct +0.40
VAL| reward -0.348559707403183, SK 0
ANSWER TO CONTEXT0: That's cool. I'm just a small chatbot. I'm really good. I'm just bit of a small chatot. I'm small chatbot and I'm a small chatbot and small chatbot. I'm just a small chatbot and chatbot. I haven't
ANSWER TO CONTEXT1: WhoY?
ANSWER TO CONTEXT2: I do that. I'm allergic to the parts of the body. Flying is awful. I do not have any huddies.
ANSWER TO CONTEXT3: Sure. We run a pool in a pool database, but if it's time for a sample information for the samples from the samples from a sample material is collected, and sample for a sample number is collected, with the sample information are collected, and samples were are collected.
 After the sample with the sample,
ANSWER TO CONTEXT4: Awww, item.
   return result
ANSWER TO CONTEXT5: Yes. You can be a pro. What kind of do you do it?
ANSWER TO CONTEXT6: I don't l

RL CYCLE|:   2%|▏         | 10/500 [00:45<30:19,  3.71s/it, R=-0.16, kl=0.000, loss=-0.197]

step 10: R -0.16  kl 0.000  length -0.39  no_end -0.10  repeat -0.06  generic +0.00  copy +0.00  start_punct -0.03  end_punct +0.42


RL CYCLE|:   4%|▍         | 20/500 [01:20<26:59,  3.37s/it, R=-0.12, kl=0.001, loss=-0.234]

...

RL CYCLE|:  96%|█████████▌| 480/500 [23:59<00:56,  2.82s/it, R=+0.49, kl=1.880, loss=-0.435]

step 480: R +0.49  kl 1.880  length -0.01  no_end +0.00  repeat +0.00  generic +0.00  copy +0.00  start_punct +0.00  end_punct +0.50


RL CYCLE|:  98%|█████████▊| 490/500 [24:22<00:20,  2.03s/it, R=+0.49, kl=0.011, loss=-0.381]

step 490: R +0.49  kl 0.011  length -0.01  no_end +0.00  repeat +0.00  generic +0.00  copy +0.00  start_punct +0.00  end_punct +0.50


RL CYCLE|: 100%|██████████| 500/500 [24:43<00:00,  2.97s/it, R=+0.48, kl=0.039, loss=-0.249]


In [19]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [20]:
def load_model(path):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model = NN(**ckpt['MODEL_SETTINGS_CONFIG']).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    return model
@torch.no_grad()
def test_loss(model):
    torch.manual_seed(0)
    data = torch.utils.data.DataLoader(
        SFTDataset(SFT_TEST_TOKENS, SFT_TEST_MASK, SFT_TEST_OFFSETS), 96, shuffle=False,
        num_workers=4, collate_fn=collate_fn, drop_last=False)
    losses = []
    for ctx, ans in data:
        ctx, ans = ctx[:, -256:].to(DEVICE), ans[:, :64].to(DEVICE)
        b = ans.shape[0]
        t = ((torch.arange(b, device=DEVICE) + torch.rand(1, device=DEVICE)) / b).clamp(min=1e-3)
        noise = torch.rand(ans.shape, device=DEVICE) < t[:, None]
        with torch.amp.autocast("cuda"):
            logits = model(ctx, torch.where(noise, MASK_ID, ans)).float()
        ce = torch.nn.functional.cross_entropy(logits.transpose(1, 2), ans, reduction="none")
        losses.append(((ce * noise).sum(1) / t / 64).mean().item())
    return sum(losses) / len(losses)

def generation_metrics(model, contexts, attempts_num=4, temperature=0.7):
    torch.manual_seed(0)

    contexts = contexts[:, -256:].to(DEVICE)
    output, output_len = rollout(model, batch_context=contexts, attempts_num=attempts_num, answer_len=64,
                                temperature=temperature, top_k=None, block_size=8, 
                                min_len=3, use_blocks=True)
    R, stats = reward(contexts.repeat_interleave(attempts_num, dim=0), output, output_len, DEVICE, answer_window=64)
    stats = {k:stats[k].mean().item() for k in stats}

    temporary = (output == EOS_ID).nonzero()
    no_end_rate = (output_len == 64).float().mean().item()
    return R.mean().item(), stats, output_len.float().mean().item(), no_end_rate

def compare(models, attempts_num):
    torch.manual_seed(0)
    rl = True
    for model in models:
        model.eval()
        D = torch.utils.data.DataLoader(SFTDataset(SFT_TEST_TOKENS, SFT_TEST_MASK, SFT_TEST_OFFSETS), 32, shuffle=False, num_workers=4, collate_fn=collate_fn, pin_memory=True, drop_last=False, persistent_workers=True)
        D = next(iter(D))
        reward, stats, output_len, no_end_rate = generation_metrics(model, D[0], attempts_num)
        L = test_loss(model)
        if rl:
            print("RL")
            rl = False
        else:
            print("SFT")
        print(reward, stats, output_len, no_end_rate)
        print(L)

rl = load_model('/kaggle/working/RL_BEST.pt')
sft = load_model('/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt')
compare([rl, sft], attempts_num=4)

RL
-0.1828737109899521 {'length': -0.1640625, 'no_end': -0.15625, 'repeat': -0.2766237258911133, 'generic': -0.0078125, 'copy': 0.0, 'start_punct': 0.0, 'end_punct': 0.421875} 22.7421875 0.15625
1.1485067009925842
SFT
-1.1496580839157104 {'length': -0.4921875, 'no_end': -0.3828125, 'repeat': -0.5051267147064209, 'generic': -0.015625, 'copy': 0.0, 'start_punct': -0.0859375, 'end_punct': 0.33203125} 34.8359375 0.3828125
0.9109293445944786
